In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, max_completion_tokens=2048).bind(logprobs=True)


In [8]:
prompt = PromptTemplate.from_template("{topic}에 대하여 3문장으로 설명해줘.")
chain = prompt | model | StrOutputParser()

answer = chain.batch([
    {"topic" : "ChatGPT"},
    {"topic" : "Instagram"},
    {"topic" : "멀티모달"},
    {"topic" : "프로그래밍"},
    {"topic" : "머신러닝"},
    ],
    config = {"max_concurrency" : 3}, #동시에 처리할 최대 작업수                   
    )

In [ ]:
print(*answer, sep="\n")

async for token in chain.astream({"topic" : "YouTube"}):
    print(token, end="", flush=True)
    

ChatGPT는 OpenAI가 개발한 대화형 인공지능 모델로, 자연어 처리 기술을 기반으로 합니다. 사용자의 질문이나 요청에 대해 인간과 유사한 방식으로 대답하며, 다양한 주제에 대해 정보를 제공할 수 있습니다. 이 모델은 사용자와의 상호작용을 통해 지속적으로 학습하고 발전하는 특징을 가지고 있습니다.
인스타그램은 사용자가 사진과 동영상을 공유하고 소통할 수 있는 소셜 미디어 플랫폼입니다. 다양한 필터와 편집 도구를 제공하여 사용자가 자신의 콘텐츠를 창의적으로 표현할 수 있도록 돕습니다. 또한, 사용자들은 친구, 가족, 유명인사 등 다양한 계정을 팔로우하며 실시간으로 업데이트를 확인할 수 있습니다.
멀티모달은 다양한 형태의 데이터를 동시에 처리하고 분석하는 기술을 의미합니다. 예를 들어, 텍스트, 이미지, 음성 등 서로 다른 모드를 결합하여 더 풍부한 정보를 추출하고 이해할 수 있습니다. 이는 인공지능 및 머신러닝 분야에서 특히 중요한 접근 방식으로, 더 나은 성능과 정확성을 제공합니다.
프로그래밍은 컴퓨터가 수행할 작업을 정의하는 과정으로, 주로 프로그래밍 언어를 사용하여 코드를 작성합니다. 이 과정에서는 알고리즘과 데이터 구조를 활용하여 문제를 해결하고, 소프트웨어나 애플리케이션을 개발합니다. 프로그래밍을 통해 사용자는 컴퓨터의 기능을 확장하고, 다양한 자동화 작업을 수행할 수 있습니다.
머신러닝은 데이터에서 패턴을 학습하여 예측이나 결정을 내리는 인공지능의 한 분야입니다. 알고리즘은 주어진 데이터를 기반으로 모델을 생성하고, 이를 통해 새로운 데이터에 대한 통찰을 제공합니다. 머신러닝은 다양한 응용 분야에서 활용되며, 예를 들어 이미지 인식, 자연어 처리, 추천 시스템 등이 있습니다.
YouTube는 사용자들이 동영상을 업로드, 공유, 시청할 수 있는 세계 최대의 비디오 플랫폼입니다. 2005년에 설립된 이래로, 다양한 콘텐츠 제작자들이 창의적인 영상을 제작하여 수익을 창출할 수 있는 기회를 제공합니다. 또한, 사용자들은 댓글, 좋아요, 구독 등을 통해 

In [14]:
my_process = chain.ainvoke({"topic" : "NVDA"})
await my_process
print("Done")
#my_process가 끝날때까지 대기

Done


In [23]:
my_abatch_process = chain.abatch(
    [{"topic" : "센트러스에너지"}, {"topic" : "삼성전기"}, {"topic" : "GraphRag"}]
)

await my_abatch_process
print("done")

done


In [31]:
from langchain_core.runnables import RunnableParallel

chain_1 = (
    PromptTemplate.from_template("{country}의 수도는 어디야?")
    |model
    |StrOutputParser()
)

chain_2 = (
    PromptTemplate.from_template("{country}의 면적은 얼마야?")
    |model
    |StrOutputParser()
)

combined = RunnableParallel(capital=chain_1, area=chain_2)

In [33]:
chain_1.invoke({"country" : "대한민국"})

'대한민국의 수도는 서울입니다.'

In [34]:
chain_2.invoke({"country" : "미국"})

'미국의 면적은 약 9,830,000 평방킬로미터(3,796,000 평방마일)입니다. 이는 세계에서 세 번째로 큰 나라에 해당합니다.'

In [35]:
combined.invoke({"country" : "대한민국"})

{'capital': '대한민국의 수도는 서울입니다.',
 'area': '대한민국의 면적은 약 100,210 평방킬로미터입니다. 이는 한반도의 남쪽 부분에 해당하며, 북한과 함께 한반도를 이루고 있습니다.'}

In [36]:
chain_1.batch([{"country" : "대한민국"}, {"country" : "미국"}])

['대한민국의 수도는 서울입니다.', '미국의 수도는 워싱턴 D.C.입니다.']

In [37]:
chain_2.batch([{"country" : "대한민국"}, {"country" : "미국"}])

['대한민국의 면적은 약 100,210 평방킬로미터(㎢)입니다. 이는 한반도의 남쪽 부분에 해당하며, 북한과의 경계를 포함한 면적입니다.',
 '미국의 면적은 약 9,830,000 제곱킬로미터(3,796,742 제곱마일)입니다. 이는 세계에서 세 번째로 큰 국가로, 러시아와 캐나다에 이어 가장 큰 면적을 가진 나라입니다.']

In [38]:
combined.batch([{"country" : "대한민국"}, {"country" : "미국"}])

[{'capital': '대한민국의 수도는 서울입니다.',
  'area': '대한민국의 면적은 약 100,210 평방킬로미터(㎢)입니다. 이는 한반도의 남쪽 부분을 차지하고 있으며, 대략적인 수치입니다.'},
 {'capital': '미국의 수도는 워싱턴 D.C.입니다.',
  'area': '미국의 총 면적은 약 9,830,000 평방킬로미터(3,796,000 평방마일)입니다. 이는 세계에서 세 번째로 큰 나라로, 러시아와 캐나다에 이어 위치하고 있습니다.'}]

In [40]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI


prompt = PromptTemplate.from_template("{num} 의 10배는?")
llm = ChatOpenAI(model="gpt-4o-mini",)

chain = prompt | llm

In [41]:
chain.invoke({"num": 5})

AIMessage(content='5의 10배는 50입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b85f3645a5', 'id': 'chatcmpl-ENer5S5FncmA8nrIimuXhWuoVn1Hm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09b06-b105-73a0-8ad9-b95164055b07-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 10, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [42]:
chain.invoke(5)

AIMessage(content='5의 10배는 50입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b85f3645a5', 'id': 'chatcmpl-ENerGJ1cqeTJ3rGVcYEbjbxEKr5lN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09b06-dabf-76f0-b28d-eca792def3d1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 10, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
from langchain_core.runnables import RunnablePassthrough

RunnablePassthrough().invoke({"num": 10})

{'num': 10}

In [45]:
runnable_chain = {"num": RunnablePassthrough()} | prompt | llm

runnable_chain.invoke(10)

AIMessage(content='10의 10배는 100입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f5dd3ac0b7', 'id': 'chatcmpl-ENeszT4K7eOoDuO1yzBdC6Bb0Kfxi', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09b08-7d4c-7963-a28a-48aba2794add-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 10, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [46]:
RunnablePassthrough().invoke({"num": 1})

{'num': 1}

In [47]:
(RunnablePassthrough.assign(new_num=lambda x: x["num"] * 3)).invoke({"num": 1})

{'num': 1, 'new_num': 3}

In [ ]:
from langchain_core.runnables import RunnableParallel

runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),
    modified=lambda x: x["num"] + 1,
)

runnable.invoke({"num": 1})

{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}